In [224]:
import pandas as pd
import numpy as np
from typing import Tuple
from scipy import stats

## 5 (c)

Use the pivot statistic above, but, instead of using the Gamma, estimate its distribution quantiles using the bootstrap - use those quantile to construct a 95% CI.

Choose λ = 3 and generate 10000 samples of 20 random variables T1,...,T20 with distribution Exp(3). For each of the 10000 samples, construct the three CIs. For each CI, calculate its coverage (the proportion of simulations where the true value is in the interval), and report the 3 numbers.

Show your work and include all the code you used.

In [225]:
SEED: int = 42

In [226]:
lambda_: float = 3.0
N: int = 10_000
n: int = 20
alpha: float = 0.05

In [227]:
rng = np.random.default_rng(seed = SEED)

In [228]:
def bootstrap_ci(data: np.ndarray, B: int = 1_000, alpha: float = 0.05) -> Tuple[float, float]:

    m = len(data)
    lambda_hat = 1 / data.mean()

    boot_samples = rng.exponential(scale = 1 / lambda_hat, size = (B, m))

    q_ = lambda_hat * boot_samples.sum(axis = 1)

    l_q = np.quantile(q_, alpha / 2)
    u_q = np.quantile(q_, 1 - alpha / 2)

    total = data.sum()
    return l_q / total, u_q / total

In [229]:
data_ = rng.exponential(scale = 1 / lambda_, size = n)

l, u = bootstrap_ci(data = data_, B = 1_000, alpha = alpha)

print(f"Bootstrap 95% CI: [{l:.4f}, {u:.4f}]")

Bootstrap 95% CI: [1.7933, 4.1790]


In [230]:
data = rng.exponential(scale = 1 / lambda_, size = (N, n))

In [231]:
# Normal Approximation

z = - stats.norm.ppf(q = alpha / 2)

lambda_hat = 1 / data.mean(axis = 1)

norm_ci = lambda_hat + np.array([
    - z * lambda_hat / np.sqrt(n), z * lambda_hat / np.sqrt(n)
]) 

norm_coverage = (((norm_ci[0] <= lambda_).astype(int) + (lambda_ <= norm_ci[1]).astype(int)) == 2).mean()

norm_coverage

0.9505

In [232]:
# Pivot

l_q = stats.gamma.ppf(q = alpha / 2, a = n, scale = 1 / 1)
u_q = stats.gamma.ppf(q = 1 - alpha / 2, a = n, scale = 1 / 1)

l_ci = (1 / data.sum(axis = 1)) * l_q
u_ci = (1 / data.sum(axis = 1)) * u_q

pivot_ci = np.array([
    l_ci, u_ci
])

pivot_coverage = (((pivot_ci[0] <= lambda_).astype(int) + (lambda_ <= pivot_ci[1]).astype(int)) == 2).mean()

pivot_coverage

0.9489

In [233]:
# Bootstrap

boot_ci = np.array([bootstrap_ci(data[i], B = 1_000, alpha = alpha) for i in range(0, N)])

boot_coverage = ((boot_ci[:, 0] <= lambda_) & (lambda_ <= boot_ci[:, 1])).mean()

boot_coverage

0.9453

In [234]:
pd.DataFrame(
    data = {
        'normal'  : [norm_coverage],
        'pivot'   : [pivot_coverage],
        'bootstrap' : [boot_coverage]
    },
    index = ['coverage']
)

,normal,pivot,bootstrap
coverage,0.9505,0.9489,0.9453
